# FastSpeech 2: Telugu Text-to-Speech Training & Evaluation
### Multilingual Non-Autoregressive TTS System (IndicTTS Telugu Dataset)

This notebook runs the complete end-to-end pipeline:
1. **Environment Setup & Dependencies**: Torch 2.x, Torchaudio, PyWorld, Librosa.
2. **Acoustic Preprocessing & Forced Alignment**: HiFi-GAN linear log-mels, continuous F0, STFT energy, Telugu Unicode Akshara decomposition, and phonetic forced alignment.
3. **FastSpeech 2 Training**: Non-autoregressive transformer with masked multi-task loss.
4. **Inference & Synthesis**: HiFi-GAN neural vocoding on the 5 benchmark test sentences (T1–T4).
5. **Interactive Audio Players & Variance Adaptor Visualizations**.

## 1. Setup Environment & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q pyworld librosa soundfile matplotlib pyyaml tqdm torch torchaudio soxr
print('✓ Telugu TTS dependencies installed successfully.')

## 2. Setup Project Repository & Unpack Telugu Dataset

In [ ]:
import os, glob, zipfile, shutil, subprocess

project_dir = '/kaggle/working/project'

# 1. Clone or pull latest codebase from GitHub
if not os.path.exists(project_dir):
    print('Cloning project from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/Tarak098/fastspeech2_project.git', project_dir], check=True)
else:
    print('Updating project repository from GitHub...')
    subprocess.run(['git', '-C', project_dir, 'pull'], check=False)

# 2. Locate and unpack Telugu dataset
data_dest = os.path.join(project_dir, 'Telugu', 'data', 'Svarah')
os.makedirs(data_dest, exist_ok=True)

dataset_zips = glob.glob('/kaggle/input/**/telugu*.zip', recursive=True) + \
               glob.glob('/kaggle/input/**/indictts*.zip', recursive=True) + \
               glob.glob('/kaggle/working/**/telugu*.zip', recursive=True)

dataset_zips = [z for z in dataset_zips if 'checkpoint' not in z.lower()]
if dataset_zips:
    print('Found Telugu dataset archive:', dataset_zips[0])
    print(f'Extracting to {data_dest}...')
    with zipfile.ZipFile(dataset_zips[0], 'r') as z:
        z.extractall(data_dest)
    print('✓ Telugu dataset extracted successfully.')
else:
    print('Checking for uncompressed dataset files in /kaggle/input...')

# 3. Ensure required directories exist
os.makedirs(os.path.join(project_dir, 'Telugu', 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(project_dir, 'Results', 'Telugu', 'test_generated_audios'), exist_ok=True)
os.makedirs(os.path.join(project_dir, 'Results', 'Telugu', 'variance_adaptor_plots', 'duration'), exist_ok=True)
os.makedirs(os.path.join(project_dir, 'Results', 'Telugu', 'variance_adaptor_plots', 'pitch'), exist_ok=True)
os.makedirs(os.path.join(project_dir, 'Results', 'Telugu', 'variance_adaptor_plots', 'energy'), exist_ok=True)
print('✓ Telugu project workspace ready!')

## 3. Run Telugu Preprocessing & Forced Alignment Pipeline
Extracts 5h Train / 1h Val, resamples audio to 22.05kHz, extracts continuous F0, energy, and genuine phonetic durations.

In [ ]:
%cd /kaggle/working/project/Telugu
!python preprocess.py config.yaml

## 4. Train / Fine-tune FastSpeech 2 (Telugu)
Trains with multi-task loss (masked Mel L1/MSE, duration MSE, pitch MSE, energy MSE). Warm-starts from existing checkpoint if available.

In [ ]:
%cd /kaggle/working/project/Telugu
!python train.py config.yaml

## 5. Benchmark Synthesis (T1–T4) with HiFi-GAN Vocoding
Synthesizes the 5 required benchmark sentences and generates duration, pitch, and energy contour plots.

In [ ]:
%cd /kaggle/working/project/Telugu
!python inference.py --config config.yaml --checkpoint checkpoints/best_model.pth

## 6. Interactive Audio Players & Variance Adaptor Visualizations
Listen directly to the generated speech and inspect acoustic variance contours.

In [ ]:
from IPython.display import Audio, display, Image
import glob, os

audio_dir = '/kaggle/working/project/Results/Telugu/test_generated_audios'
audio_files = sorted(glob.glob(os.path.join(audio_dir, '*.wav')))

print(f'=== Synthesized {len(audio_files)} Benchmark Telugu Utterances ===\n')
for af in audio_files:
    sid = os.path.splitext(os.path.basename(af))[0]
    print(f'\n============================================================')
    print(f' Utterance ID: {sid}')
    print(f' Audio File: {af}')
    print(f'============================================================')
    display(Audio(af))
    
    # Display duration and pitch plots
    d_plot = f'/kaggle/working/project/Results/Telugu/variance_adaptor_plots/duration/{sid}_duration.png'
    p_plot = f'/kaggle/working/project/Results/Telugu/variance_adaptor_plots/pitch/{sid}_pitch.png'
    e_plot = f'/kaggle/working/project/Results/Telugu/variance_adaptor_plots/energy/{sid}_energy.png'
    
    if os.path.exists(d_plot):
        print('Duration Alignment:')
        display(Image(d_plot, width=600))
    if os.path.exists(p_plot):
        print('Pitch Contour:')
        display(Image(p_plot, width=600))
    if os.path.exists(e_plot):
        print('Energy Contour:')
        display(Image(e_plot, width=600))

## 7. Interactive Arbitrary Telugu Sentence Synthesis

In [ ]:
%cd /kaggle/working/project/Telugu
!python inference.py --text "నమస్కారం, మీరు ఎలా ఉన్నారు? తెలుగు ఫాస్ట్‌స్పీచ్ 2 సంశ్లేషణ ఇప్పుడు నిజమైన ఫోనెటిక్ అలైన్‌మెంట్ మరియు హైఫై-గాన్ న్యూరల్ వోకోడర్‌తో అద్భుతంగా పనిచేస్తోంది." --output /kaggle/working/custom_telugu.wav

from IPython.display import Audio
Audio('/kaggle/working/custom_telugu.wav')

## 8. Package All Deliverables for Submission Download

In [ ]:
%cd /kaggle/working
!zip -r telugu_results.zip project/Results/Telugu project/Telugu/checkpoints
print('✓ telugu_results.zip is ready for download in Kaggle Output tab!')